# **Homework 2: Apriori**

Deadline: Friday 20th February 2026

Name: Aleena Zahra 23i-2514 DS-B

## Overview: 
The objective of this homework is to perform Market Basket Analysis using the Apriori 
algorithm and extract association rules from transactional data. 
Dataset: 
You are given a dataset titled online_retail.csv. 


In [21]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules, apriori
import numpy as np


In [4]:
df = pd.read_csv('Aleena Zahra - online_retail - online_retail.csv')


In [5]:
df.head()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


### Basic Preprocessing and Cleaning

In [6]:
# remove cancelled transactions
df = df[df['Quantity'] > 0]
# remove null values 
df = df.dropna(subset=['Description', 'InvoiceDate', 'Quantity'])

# drop unit price,country and quantity columns
df = df.drop(columns=['UnitPrice', 'Country', 'Quantity'])


In [7]:

# lowercase description column because it bothers me
df['Description'] = df['Description'].str.lower()


## **Tasks**

## 1. Transaction Construction 

● Construct transactions such that each invoice corresponds to one basket. 

● Represent each basket as a list of purchased items. 


In [8]:
# get the number of unique invoiceDates
print(df['InvoiceDate'].nunique())



32


In [9]:
baskets = (
    df.groupby('InvoiceDate')['Description']
      .apply(list)
      .reset_index(name='items')
)
print(len(baskets))

32



## 2. Time Based Baskets 
● Extract hour from InvoiceDate 

● Divide transactions into 4 groups: Morning, Afternoon, Evening, Night 

● Create 4 separate basket collections based on these time groups. 


In [10]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Hour'] = df['InvoiceDate'].dt.hour

# Define time groups
def get_time_period(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['TimePeriod'] = df['Hour'].apply(get_time_period)

In [11]:
baskets_by_time = {}

for period in ['Morning', 'Afternoon', 'Evening', 'Night']:
    subset = df[df['TimePeriod'] == period]
    baskets_by_time[period] = (
        subset.groupby('InvoiceDate')['Description']
              .apply(list)
              .tolist()                     # list of lists
    )
    
    print(f"{period}: {len(baskets_by_time[period])} transactions")

Morning: 32 transactions
Afternoon: 0 transactions
Evening: 0 transactions
Night: 0 transactions



## 3. One Hot Encoding 
● Write your own function that converts a list of baskets into a one hot encoded 
Dataframe. 
● Use a library-based encoder to do the same. 
● Compare both encodings (dimensions and example rows). 


In [12]:
def manual_onehot(baskets_list):
    all_items = sorted(set(item for basket in baskets_list for item in basket))
    item_to_idx = {item: i for i, item in enumerate(all_items)}
    
    matrix = np.zeros((len(baskets_list), len(all_items)), dtype=bool)
    
    for row_idx, basket in enumerate(baskets_list):
        for item in basket:
            col_idx = item_to_idx[item]
            matrix[row_idx, col_idx] = True
            
    df_onehot = pd.DataFrame(matrix, columns=all_items)
    return df_onehot, all_items


morning_baskets = baskets_by_time['Morning']

# Manual
manual_df, items_manual = manual_onehot(morning_baskets)

In [13]:
te = TransactionEncoder()
te_ary = te.fit(morning_baskets).transform(morning_baskets)
lib_df = pd.DataFrame(te_ary, columns=te.columns_)

print("Shape manual :", manual_df.shape)
print("Shape library:", lib_df.shape)
print("Columns equal?", list(manual_df.columns) == list(lib_df.columns))

Shape manual : (32, 277)
Shape library: (32, 277)
Columns equal? True



## 4. Frequent Itemset Mining 
● Apply the Apriori algorithm one each time-based basket. 
● Use at least 2 different support thresholds. 
● Report: 
○ Number of frequent itemsets. 
○ Largest itemset length. 


In [14]:
from mlxtend.frequent_patterns import apriori

results = {}

for period in baskets_by_time:
    if len(baskets_by_time[period]) < 5:  
        print(f"Skipping {period} — too few baskets")
        continue
        
    te = TransactionEncoder()
    te_ary = te.fit(baskets_by_time[period]).transform(baskets_by_time[period])
    df_onehot = pd.DataFrame(te_ary, columns=te.columns_)
    
    print(f"\n{period} — {len(df_onehot):,} transactions, {len(te.columns_):,} items")
    
    for minsup in [0.08, 0.04]:  
        frequent_itemsets = apriori(df_onehot, min_support=minsup, use_colnames=True)
        frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
        
        results[(period, minsup)] = frequent_itemsets
        
        print(f"  minsup = {minsup:>5.3f} → {len(frequent_itemsets):>4} itemsets | "
              f"max length = {frequent_itemsets['length'].max()}")


Morning — 32 transactions, 277 items
  minsup = 0.080 → 65551 itemsets | max length = 16
  minsup = 0.040 → 65621 itemsets | max length = 16
Skipping Afternoon — too few baskets
Skipping Evening — too few baskets
Skipping Night — too few baskets



## 5. Association Rule Mining 
● Extract rules using Confidence and Lift. 
● Filter rules with Confidence >= 0.6 and Lift > 1 


In [ ]:
all_rules_list = []

for minsup in [0.08, 0.04]:  
    # Apriori mit max len=3
    frequent_itemsets = apriori(df_onehot, min_support=minsup, use_colnames=True, max_len=3)
    results[(period, minsup)] = frequent_itemsets
    print(f"  minsup = {minsup:>5.3f} → {len(frequent_itemsets):>5} itemsets")

for (period, minsup), freq_itemsets in results.items():
    if freq_itemsets.empty:
        continue
        
    # Regeln generieren
    rules = association_rules(freq_itemsets, metric="confidence", min_threshold=0.6)
    
    if not rules.empty:
        rules = rules[rules['lift'] > 1.0].copy()
        
        rules['period'] = period
        rules['min_support'] = minsup
        
        all_rules_list.append(rules)


In [ ]:

if all_rules_list:
    final_rules_df = pd.concat(all_rules_list, ignore_index=True)
    
    cols = ['period', 'min_support', 'antecedents', 'consequents', 'support', 'confidence', 'lift']
    final_rules_df = final_rules_df[cols].sort_values(['period', 'lift'], ascending=[True, False])
    
    print("\n~~~~~ Finale Tabelle ~~~~~")
    print(final_rules_df.head(10)) 


## 6. Comparative Analysis 
Answer the following: 
● Which time period produces the strongest rules? 

<span style="color:red">-> Morning as all baskets are from morning</span>

● Which item pair appears in multiple time periods? 

<span style="color:red">-> None of them as all of them are in the morning</span>.


● How does changing support affect the number of rules? 

<span style="color:red">-> As support increases the number of rules decrease </span>